<a href="https://colab.research.google.com/github/Areeba-Kh571/flyrank-ml-internship-areeba/blob/main/w04_baseline_score_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Areeba-Kh571/flyrank-ml-internship-areeba/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule, in plain words:** a page is worth flagging for refresh review if it's **stale**
(created 180+ days before the decision point) and still **visible** (at least 500 impressions in
the trailing 90-day feature window) -- and among pages that clear both bars, the ones whose recent
30-day pace is already falling get flagged first, because real traffic that's actively slipping is
worth more attention than real traffic that's flat or growing. This is the same staleness +
visibility rule I piloted by hand in the Week 1 case study (180 days / 500 impressions), now coded
as a transparent score with a reason code instead of a one-off check.

**Two signals this rule leans on, checked before I trust them:**

- **Staleness** (`content_age_days`, from `dim_content.content_created_date`) -- this is the signal
  behind FlyRank's own refresh flags. I first tried `content_updated_date` instead (closer to my Week 1
  case study's literal "not updated in 180+ days"), but that field turned out to be leaky: `dim_content`
  stores each item's *current* state, not its history, so `content_updated_date` can reflect a touch that
  happened months after my March 2026 decision point -- for ~75% of rows it produced a negative
  "days since update," i.e. it was telling me about updates that hadn't happened yet as of the decision
  point. `content_created_date` doesn't have that problem (a page's creation date never changes), so it's
  the point-in-time-safe signal I actually check below, even though it's a weaker proxy for "stale" than
  true update recency would be. That trade-off -- and the leaky field I ruled out -- gets called out again
  in section 4.
- **Volume** (`imp_90d`) -- this is the signal behind FlyRank's `is_quick_win` logic (a quick win
  needs real existing traffic to be worth fixing). I check whether pages with more impressions
  behave differently on the decline label than low-impression pages.

Both checks below use a **bucket table with n printed per bucket** and the same
`is_declining_next30` label from the ML-04 data contract (Q1 2026 feature window, April 2026 label
window -- decline = the label window's impressions come in under 80% of the feature window's last
30 days). The verdict word (CONFIRMED / OPPOSITE / MIXED / FALSE) is computed by the code from the
real bucket rates, not asserted by me -- see the `verdict()` function below.

**Reason codes** (exactly one per row, in priority order):

1. `NOT_STALE` -- content_age_days < 180. No refresh case yet regardless of traffic.
2. `NOT_VISIBLE` -- stale, but imp_90d < 500. Not enough real traffic to justify review time.
3. `STALE_VISIBLE_DECLINING` -- stale, visible, and trend_ratio_90d < 0.8 (already slipping).
4. `STALE_VISIBLE_STABLE` -- stale, visible, but momentum is flat or up.

**No future-window or label-derived inputs:** the rule's inputs (`content_age_days`, `imp_90d`,
`trend_ratio_90d`) are all computed only from the Q1 feature window or `dim_content` metadata that
predates it. `is_declining_next30` and `imp_label30` (April, the label window) are used ONLY to
check the signals and evaluate the rule afterward -- never as inputs to the score. The leakage-check
cell in section 4 asserts this in code, not just in this paragraph.

In [5]:
%pip -q install duckdb

import os, getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
}
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

# Same Q1 2026 -> April 2026 windows as the ML-04 data contract, for a consistent label.
FEATURE_START = "DATE '2026-01-01'"
FEATURE_END   = "DATE '2026-03-31'"
LABEL_START   = "DATE '2026-04-01'"
LABEL_END     = "DATE '2026-04-30'"

frame = con.sql(f"""
    WITH eligible_clients AS (
        SELECT client_hash_id
        FROM {TABLES['dim_clients']}
        WHERE gsc_data_start <= {FEATURE_START}
    ),
    feat AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions)                                                                AS imp_90d,
               AVG(CASE WHEN f.gsc_avg_position > 0 THEN f.gsc_avg_position END)                      AS pos_90d,
               SUM(CASE WHEN f.report_date >  {FEATURE_END} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= {FEATURE_END} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_first60
        FROM {FACT} f
        JOIN eligible_clients c USING (client_hash_id)
        WHERE f.report_date BETWEEN {FEATURE_START} AND {FEATURE_END}
        GROUP BY 1, 2
        HAVING imp_90d >= 100
    ),
    label AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_label30
        FROM {FACT}
        WHERE report_date BETWEEN {LABEL_START} AND {LABEL_END}
        GROUP BY 1, 2
    ),
    content_age AS (
        -- staleness signal: content_created_date, NOT content_updated_date.
        -- dim_content stores current state only -- content_updated_date can reflect a touch that
        -- happened AFTER this feature window's decision point (future leakage). Creation date is
        -- immutable once set, so it's the point-in-time-safe choice even though it's a weaker proxy
        -- for "stale" than true update recency. See section 4 for the leaky version I ruled out.
        SELECT content_hash_id,
               DATE_DIFF('day', CAST(content_created_date AS DATE), {FEATURE_END}) AS content_age_days
        FROM {TABLES['dim_content']}
    )
    SELECT f.*,
           COALESCE(l.imp_label30, 0)                                             AS imp_label30,
           f.imp_last30 / NULLIF(f.imp_first60 / 2.0, 0)                          AS trend_ratio_90d,
           CASE WHEN COALESCE(l.imp_label30, 0) < 0.8 * f.imp_last30 THEN 1 ELSE 0 END AS is_declining_next30,
           ca.content_age_days
    FROM feat f
    LEFT JOIN label l USING (client_hash_id, content_hash_id)
    LEFT JOIN content_age ca USING (content_hash_id)
""").df()

print(f"{len(frame):,} content items with a full Q1 window, an eligible client, and imp_90d >= 100")
print("nulls in content_age_days:", frame['content_age_days'].isna().sum(), "-- drop these before bucketing if > 0")
frame = frame.dropna(subset=['content_age_days']).copy()

base_rate = frame['is_declining_next30'].mean()
print(f"base rate (share declining next 30 days, whole frame): {base_rate:.3f}\n")


def verdict(rates_by_bucket_ordered):
    """CONFIRMED / OPPOSITE / MIXED / FALSE from the real bucket rates -- not asserted, computed."""
    rates = list(rates_by_bucket_ordered)
    diff = rates[-1] - rates[0]
    increasing = all(b - a >= -0.02 for a, b in zip(rates, rates[1:]))
    decreasing = all(b - a <=  0.02 for a, b in zip(rates, rates[1:]))
    if abs(diff) < 0.03:
        return "FALSE"          # buckets barely differ -- the signal doesn't discriminate here
    if diff > 0 and increasing:
        return "CONFIRMED"      # rises step by step in the hypothesized direction
    if diff < 0 and decreasing:
        return "OPPOSITE"       # moves, but the wrong way -- a clean negative, still useful
    return "MIXED"              # moves, but not consistently -- don't lean on this alone


# --- Signal A: staleness (behind FlyRank's refresh flags) ---
age_bins = [-1, 90, 180, 365, 10**6]
age_labels = ['<90', '90-180', '180-365', '365+']
frame['age_bucket'] = pd.cut(frame['content_age_days'], bins=age_bins, labels=age_labels)
sig_a = frame.groupby('age_bucket', observed=True).agg(
    n=('is_declining_next30', 'size'), decline_rate=('is_declining_next30', 'mean')
)
print("SIGNAL A -- staleness (content_age_days) vs. decline rate")
print(f"bucketed rows: {sig_a['n'].sum():,} of {len(frame):,} -- a big gap here means values fell outside the bins (check for leakage/negatives before trusting the verdict)")
print(sig_a)
verdict_a = verdict(sig_a['decline_rate'].tolist())
print(f"n per bucket printed above. VERDICT A: {verdict_a}\n")

# --- Signal B: volume (behind FlyRank's quick-win logic) ---
vol_bins = [-1, 0, 499, 4999, 10**9]
vol_labels = ['0', '1-499', '500-4999', '5000+']
frame['vol_bucket'] = pd.cut(frame['imp_90d'], bins=vol_bins, labels=vol_labels)
sig_b = frame.groupby('vol_bucket', observed=True).agg(
    n=('is_declining_next30', 'size'), decline_rate=('is_declining_next30', 'mean')
)
print("SIGNAL B -- volume (imp_90d) vs. decline rate")
print(sig_b)
verdict_b = verdict(sig_b['decline_rate'].tolist())
print(f"n per bucket printed above. VERDICT B: {verdict_b}")
print("\nA FALSE or OPPOSITE verdict here is a legitimate result, not a failed cell -- it means")
print("that signal shouldn't carry weight in the rule below, and that's worth knowing before coding it in.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

102,769 content items with a full Q1 window, an eligible client, and imp_90d >= 100
nulls in content_age_days: 0 -- drop these before bucketing if > 0
base rate (share declining next 30 days, whole frame): 0.494

SIGNAL A -- staleness (content_age_days) vs. decline rate
bucketed rows: 102,769 of 102,769 -- a big gap here means values fell outside the bins (check for leakage/negatives before trusting the verdict)
                n  decline_rate
age_bucket                     
<90         21725      0.457629
90-180      15777      0.586233
180-365     48494      0.482864
365+        16773      0.486615
n per bucket printed above. VERDICT A: FALSE

SIGNAL B -- volume (imp_90d) vs. decline rate
                n  decline_rate
vol_bucket                     
1-499       32532      0.474640
500-4999    46192      0.519332
5000+       24045      0.471574
n per bucket printed above. VERDICT B: FALSE

A FALSE or OPPOSITE verdict here is a legitimate result, not a failed cell -- it means
that si

## 2. Build the ranked queue (writes the CSV)

Score, one reason code, one action label per row -- readable on purpose, no fitted weights:
`score = stale * visible * imp_90d * (2 if declining_now else 1)`. Doubling for an
already-declining page is the only "weight" in this rule, and it's there so refresh_now candidates
naturally outrank monitor candidates with similar traffic, not because a model told me to.

In [6]:
STALE_DAYS = 180
VISIBLE_IMP = 500

frame['stale'] = (frame['content_age_days'] >= STALE_DAYS).astype(int)
frame['visible'] = (frame['imp_90d'] >= VISIBLE_IMP).astype(int)
frame['declining_now'] = (frame['trend_ratio_90d'] < 0.8).astype(int)


def reason_code(row):
    if row['stale'] == 0:
        return 'NOT_STALE'
    if row['visible'] == 0:
        return 'NOT_VISIBLE'
    if row['declining_now'] == 1:
        return 'STALE_VISIBLE_DECLINING'
    return 'STALE_VISIBLE_STABLE'


frame['reason_code'] = frame.apply(reason_code, axis=1)
frame['score'] = frame['stale'] * frame['visible'] * frame['imp_90d'] * (1 + frame['declining_now'])

ACTION = {
    'STALE_VISIBLE_DECLINING': 'refresh_now',
    'STALE_VISIBLE_STABLE': 'monitor',
    'NOT_STALE': 'no_action',
    'NOT_VISIBLE': 'no_action',
}
frame['action'] = frame['reason_code'].map(ACTION)

print("reason_code counts:")
print(frame['reason_code'].value_counts())
print("\naction counts:")
print(frame['action'].value_counts())

queue_cols = ['client_hash_id', 'content_hash_id', 'score', 'reason_code', 'action',
              'imp_90d', 'pos_90d', 'content_age_days', 'trend_ratio_90d']
queue = frame.sort_values('score', ascending=False)[queue_cols].reset_index(drop=True)

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"\nwrote work/outputs/baseline_action_score.csv -- {len(queue):,} rows")
queue.head(10)

reason_code counts:
reason_code
NOT_STALE                  37502
STALE_VISIBLE_STABLE       34823
NOT_VISIBLE                20568
STALE_VISIBLE_DECLINING     9876
Name: count, dtype: int64

action counts:
action
no_action      58070
monitor        34823
refresh_now     9876
Name: count, dtype: int64

wrote work/outputs/baseline_action_score.csv -- 102,769 rows


,client_hash_id,content_hash_id,score,reason_code,action,imp_90d,pos_90d,content_age_days,trend_ratio_90d
0,client_73cda7b4e4f265ea,content_e241d6415ac9e534,1019368.0,STALE_VISIBLE_DECLINING,refresh_now,509684.0,3.204471,412,0.753955
1,client_e547b89c05043229,content_eadb33b5df496f4a,830289.0,STALE_VISIBLE_STABLE,monitor,830289.0,2.497799,375,5.649965
2,client_73cda7b4e4f265ea,content_4d0d79fc12632ef8,777474.0,STALE_VISIBLE_DECLINING,refresh_now,388737.0,4.069617,243,0.377073
3,client_e547b89c05043229,content_1e921148b5fee86a,768278.0,STALE_VISIBLE_DECLINING,refresh_now,384139.0,4.775369,467,0.190524
4,client_73cda7b4e4f265ea,content_cf651123f1085418,721644.0,STALE_VISIBLE_DECLINING,refresh_now,360822.0,6.046196,412,0.742830
5,client_e547b89c05043229,content_c9a0c2fdbdbfb562,704384.0,STALE_VISIBLE_DECLINING,refresh_now,352192.0,2.139233,375,0.418303
6,client_73cda7b4e4f265ea,content_db122b8ba22641b8,617422.0,STALE_VISIBLE_DECLINING,refresh_now,308711.0,4.345542,187,0.791075
7,client_73cda7b4e4f265ea,content_c19eed2225ee5f40,602682.0,STALE_VISIBLE_DECLINING,refresh_now,301341.0,3.489294,243,0.626752
8,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,559094.0,STALE_VISIBLE_DECLINING,refresh_now,279547.0,13.687138,243,0.488623
9,client_73cda7b4e4f265ea,content_29c4a3831609805d,510940.0,STALE_VISIBLE_DECLINING,refresh_now,255470.0,1.798399,260,0.548545


## 3. Top-10 review

For each of the top 10: the action, why it's there, and what would make it wrong -- generated from
the real reason code and the real feature values for that row, not written by hand in advance.

In [7]:
WHY = {
    'STALE_VISIBLE_DECLINING': "stale ({age}d old) and still visible ({imp:,.0f} impr/90d), and recent momentum is already falling (ratio {tr:.2f}).",
    'STALE_VISIBLE_STABLE':    "stale ({age}d old) and still visible ({imp:,.0f} impr/90d), but momentum is flat/up (ratio {tr:.2f}) -- lower urgency than a declining page.",
}
WRONG = {
    'STALE_VISIBLE_DECLINING': "wrong if this dip is seasonal or expected for this content rather than real decay -- check its history before spending refresh time on it.",
    'STALE_VISIBLE_STABLE':    "wrong if this is intentionally stable evergreen content that never needed a refresh in the first place.",
}

top10 = queue.head(10)
for i, r in top10.iterrows():
    why = WHY.get(r['reason_code'], 'n/a').format(age=r['content_age_days'], imp=r['imp_90d'], tr=r['trend_ratio_90d'])
    wrong = WRONG.get(r['reason_code'], 'n/a')
    print(f"{i+1}. [{r['action']}] {why}")
    print(f"    what would make it wrong: {wrong}\n")

1. [refresh_now] stale (412d old) and still visible (509,684 impr/90d), and recent momentum is already falling (ratio 0.75).
    what would make it wrong: wrong if this dip is seasonal or expected for this content rather than real decay -- check its history before spending refresh time on it.

2. [monitor] stale (375d old) and still visible (830,289 impr/90d), but momentum is flat/up (ratio 5.65) -- lower urgency than a declining page.
    what would make it wrong: wrong if this is intentionally stable evergreen content that never needed a refresh in the first place.

3. [refresh_now] stale (243d old) and still visible (388,737 impr/90d), and recent momentum is already falling (ratio 0.38).
    what would make it wrong: wrong if this dip is seasonal or expected for this content rather than real decay -- check its history before spending refresh time on it.

4. [refresh_now] stale (467d old) and still visible (384,139 impr/90d), and recent momentum is already falling (ratio 0.19).
    w

## 4. Weak picks + leakage check

Which picks look wrong, and why -- plus a code-enforced check (not just a claim) that no product
flag or future-window column ever entered the score.

**The leaky signal I ruled out:** my first version of section 1 used `content_updated_date` for
staleness instead of `content_created_date`. On the real warehouse, that produced a negative
"days since update" for roughly 75% of rows -- proof that the field reflects touches that happened
*after* my March 2026 decision point, not before it. `dim_content` is a current-state dimension
table, not a historical log, so anything from it that can change over time (like an update
timestamp) is unsafe to use as a backward-looking feature. Only `content_created_date` (fixed at
creation, never changes) is point-in-time-safe. I caught this from the real run's numbers
(99.4% of rows reading `NOT_STALE`, and most rows missing from the signal-A bucket table
entirely) before it went into the score -- it never touched the final rule.

In [8]:
# Leakage check: none of the score's inputs are a product decision flag or a future-window column.
BANNED = {'imp_label30', 'is_declining_next30', 'trend_direction', 'health_score',
          'priority_score', 'action_type', 'refresh_tier', 'needs_ctr_fix', 'is_quick_win'}
score_inputs = {'stale', 'visible', 'imp_90d', 'declining_now'}
assert not (score_inputs & BANNED), f"leakage: score used a banned column: {score_inputs & BANNED}"
print("leakage check passed -- score inputs are:", score_inputs)
print("(is_declining_next30 / imp_label30 were used only to CHECK the signals and rule above, never to score.)")

# Weak-pick scan: monitor-labeled pages that are already ranking well (avg position in top 3)
# don't obviously need a refresh -- worth a second look even though the rule flagged them stale+visible.
weak = queue[(queue['action'] == 'monitor') & (queue['pos_90d'] <= 3)]
print(f"\npotential weak picks (action=monitor, already ranking top-3 on average): {len(weak)} rows")
if len(weak):
    print(weak.head(5))
else:
    print("none found in this run -- worth re-checking if the rule's thresholds change later.")

leakage check passed -- score inputs are: {'stale', 'visible', 'imp_90d', 'declining_now'}
(is_declining_next30 / imp_label30 were used only to CHECK the signals and rule above, never to score.)

potential weak picks (action=monitor, already ranking top-3 on average): 2919 rows
             client_hash_id           content_hash_id     score  \
1   client_e547b89c05043229  content_eadb33b5df496f4a  830289.0   
10  client_e547b89c05043229  content_ec2e0346994fb5a5  509532.0   
33  client_73cda7b4e4f265ea  content_6302b8bce0bb84cb  365677.0   
55  client_e547b89c05043229  content_4ffe18112a5642e3  299924.0   
85  client_73cda7b4e4f265ea  content_17494d099b0a537e  255403.0   

             reason_code   action   imp_90d   pos_90d  content_age_days  \
1   STALE_VISIBLE_STABLE  monitor  830289.0  2.497799               375   
10  STALE_VISIBLE_STABLE  monitor  509532.0  2.646690               434   
33  STALE_VISIBLE_STABLE  monitor  365677.0  2.520709               243   
55  STALE_VISIBLE_

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
